In [1]:
import requests, zipfile, io

files_url = "https://ideami.com/llm_train"

response = requests.get(files_url)
zipfile.ZipFile(io.BytesIO(response.content)).extractall(".")

!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.8 MB/s eta 0:00:00


In [2]:
import os, sys
import ipdb
from tqdm import tqdm
from datetime import datetime
import platform, shutil

import torch
import torch.nn as nn
from torch.nn import functional as F

#tokenizer
import sentencepiece as spm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.cuda.empty_cache()

In [3]:
# Architecture parameters

batch_size = 128
context = 512
embeded_size = 384
n_layers = 7
n_heads = 7
BIAS = True

In [4]:
# Hyperparameters

lr = 3e-4
dropout = 0.05
weight_decay = 0.01
grad_clip = 1.0

In [5]:
# Training parameters

train_iters = 100000
eval_interval = 50
eval_iters = 10
compile = False
checkpoint_dir = 'models/'
checkpoint_fn ='latest.pt'
checkpoint_load_fn = 'latest.pt'
dtype = torch.bfloat16

# Mode
inference = False

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: ", device)

Device:  cuda


In [6]:
# Logging

wandb_log = True
wandb_project = "llm1"
wandb_run_name = "llm1-" + datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

if wandb_log:
  import wandb
  wandb.init(project = wandb_project, name = wandb_run_name)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ai7532656 (ai7532656-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
with open('wiki.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  print(text[10000:10300])

 that was used to represent a team in an old TV show, The A-Team. A capital a is written "A". Use a capital A at the start of a sentence if writing.

A is also a musical note, sometimes referred to as "La".

The letter 'A' was in the Phoenician alphabet's aleph. This symbol came from a simple pictur


In [8]:
# Tokenizer

sp = spm.SentencePieceProcessor(model_file='wiki_tokenizer.model')
vocab_size = sp.get_piece_size()
print(f"Tokenizer vocab size: {vocab_size}")

Tokenizer vocab size: 4096


In [9]:
encode = lambda s: sp.encode(s)
decode = lambda l: sp.decode(l)

print(encode("Once upon a time!"))

[612, 370, 698, 265, 261, 684, 36]


In [10]:
if os.path.exists("encoded_data.pt"):
  print("Loading encoding!")
  data = torch.load('encoded_data.pt')
else:
  data = torch.tensor(encode(text), dtype = torch.long)
  torch.save(data, "encoded_data.pt")

Loading encoding!


In [11]:
data_size = len(data)
spl = int(0.9 * data_size)
train_data = data[:spl]
val_data = data[spl:]

print("Training data: " + str(data_size))

Training data: 59211077


In [12]:
def get_batch(split):
  data = train_data if split == "train" else val_data
  inds = torch.randint(len(data)-context, (batch_size,))
  x = torch.stack([data[i: i+context] for i in inds])
  y = torch.stack([data[i+1: i+context+1] for i in inds])
  x,y = x.to(device), y.to(device)
  return x,y

x,y = get_batch("train")
print(x.shape, y.shape)
print(x[0][:10])
print(y[0][:10])

torch.Size([128, 512]) torch.Size([128, 512])
tensor([1148,  486,  300, 3768,  264,  270, 2480,  280,  604, 1318],
       device='cuda:0')
tensor([ 486,  300, 3768,  264,  270, 2480,  280,  604, 1318, 1359],
       device='cuda:0')


In [14]:
class GPT(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedings = nn.Embedding(vocab_size, embeded_size)
    self.positions = nn.Embedding(context, embeded_size)
    #self.blocks = nn.Sequential(*[Block(n_heads) for _ in range(n_layers)])
    self.ln = nn.LayerNorm(embeded_size)
    self.final_linear_layer = nn.Linear(embeded_size, vocab_size, bias = BIAS)
    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)



  def forward(self, input, target= None):
    loss = None
    BS, SL = input.shape
    emb = self.embedings(input)
    pos = self.positions(torch.arrange(SL, device=device))
    x = emb + pos
    #x = self.blocks(x)
    x = self.ln(x)
    logits = self.final_linear_layer(x)

    if target is not None:
      BS,SL, VS = logits.shape
      logits = logits.view(BS*SL, VS)
      targets = targets.view(BS*SL)
      loss = F.cross_entropy(logits, targets)

      #Manual calculation

      counts = logits.exp()
      prob = counts / counts.sum(-1, keepdim = True)
      loss2 = - prob[torch.arange(BS*SL), target].log().mean()

      return logits, loss



  def generate(self, input, max=500):
    for _ in range(max):
      input = input[:,-context:]
      logits, _ = self(input)
      logits = logits[:,-1,:]
      probs = F.softmax(logits, dim=-1)
      next = torch.multinomial(probs, num_samples=1)
      input = torch.cat((input, next), dim=1)
    return input

In [16]:
x,y = get_batch("train")

model = GPT()
model = model.to(dtype)
model = model.to(device)

@torch.no_grad()
def generate_sample(input):
  t1 = torch.tensor(encode(input), dtype=torch.long, device=device)
  t1 = t1[None, :]
  newgen = model.generate(t1, max=64)[0].tolist()
  result = decode(newgen)
  print(f"{result}")

generate_sample("Once upon a time")

AttributeError: 'GPT' object has no attribute 'generate'